In [ ]:
import os, time
from pathlib import Path
import torch
import psutil
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

%matplotlib inline

display(Markdown("## Part 0 — Setup"))
display(Markdown("_Imports, helpers, RAM tracker, and the model ID. Nothing is downloaded or loaded yet._"))

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["font.size"] = 10

CATEGORY_COLORS = {
    "weights":          "#d62728",
    "tokenizer":        "#2ca02c",
    "model-config":     "#1f77b4",
    "metadata/license": "#9467bd",
    "other":            "#7f7f7f",
}

PROC = psutil.Process(os.getpid())
def ram_mb(): return PROC.memory_info().rss / (1024 ** 2)

def fmt_size(b):
    if b < 1024:        return f"{b} B"
    if b < 1024**2:     return f"{b/1024:.2f} KB"
    if b < 1024**3:     return f"{b/1024**2:.2f} MB"
    return f"{b/1024**3:.2f} GB"

STAGE_LOG = []

class Stage:
    """Reusable timing + RAM tracker. Use in every level."""
    def __init__(self, name, reads=""):
        self.name, self.reads = name, reads
    def __enter__(self):
        print(f"▶ {self.name}")
        if self.reads: print(f"  reads: {self.reads}")
        self.ram_start = ram_mb()
        self.t_start = time.time()
        return self
    def __exit__(self, *a):
        dt = time.time() - self.t_start
        ram_end = ram_mb()
        delta = ram_end - self.ram_start
        STAGE_LOG.append({
            "stage":         self.name,
            "reads":         self.reads,
            "ram_before_MB": round(self.ram_start, 1),
            "ram_after_MB":  round(ram_end, 1),
            "ram_delta_MB":  round(delta, 1),
            "time_s":        round(dt, 2),
        })
        print(f"  ✓ {dt:.2f}s   ΔRAM +{delta:.1f} MB   total {ram_end:.1f} MB\n")

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@                PART 00: SETUP COMPLETE                   @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("========== ENVIRONMENT ==========")
print("model id   :", MODEL_ID)
print("torch ver  :", torch.__version__)
print("cuda avail :", torch.cuda.is_available())
print("baseline RAM (notebook only) :", f"{ram_mb():.1f} MB")
print("=================================")

In [ ]:
display(Markdown("## Part 1 — Download stage"))
display(Markdown("_Where do files land on disk? What is fetched and in what order?_"))

from huggingface_hub import snapshot_download, scan_cache_dir

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@               PART 01: DOWNLOAD STAGE                    @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

def cache_info(model_id):
    try:
        for r in scan_cache_dir().repos:
            if r.repo_id == model_id:
                return True, r.size_on_disk
    except Exception: pass
    return False, 0

cached_before, size_before = cache_info(MODEL_ID)
display(Markdown(f"**Cache before:** hit=`{cached_before}`" +
                 (f", size=`{fmt_size(size_before)}`" if cached_before else "")))

print("========== STEP 1: CACHE CHECK ==========")
print("cache hit :", cached_before)
if cached_before:
    print("cached size :", fmt_size(size_before))
print("=========================================")

print()
print("========== STEP 2: SNAPSHOT DOWNLOAD ==========")
with Stage("snapshot_download", reads="huggingface hub"):
    local_path = snapshot_download(repo_id=MODEL_ID)
print("local path :", local_path)
print("===============================================")

hf_root = Path(os.path.expanduser("~/.cache/huggingface/hub"))
display(Markdown(f"""
**Cache layout**
- root: `{hf_root}`
- snapshot: `{local_path}`
- relative: `{Path(local_path).relative_to(hf_root)}`
"""))

def categorize(name):
    n = name.lower()
    if n.endswith((".safetensors", ".bin")):                return "weights"
    if n in ("config.json", "generation_config.json"):      return "model-config"
    if "tokenizer" in n or n == "special_tokens_map.json":  return "tokenizer"
    if n.endswith((".md", ".txt")):                         return "metadata/license"
    return "other"

rows = []
for f in os.listdir(local_path):
    full = os.path.join(local_path, f)
    if os.path.isfile(full) or os.path.islink(full):
        s = os.path.getsize(full)
        rows.append({"filename": f, "category": categorize(f),
                     "size_bytes": s, "size": fmt_size(s)})

df_files = pd.DataFrame(rows).sort_values("size_bytes").reset_index(drop=True)
df_files.index = range(1, len(df_files) + 1)
df_files.index.name = "#"

display(Markdown("**Files on disk (small → large = fetch order)**"))
display(df_files[["filename", "category", "size"]])

print()
print("========== STEP 3: FILES ON DISK ==========")
print(f"file count : {len(df_files)}")
print(f"total size : {fmt_size(df_files['size_bytes'].sum())}")
for _, row in df_files.iterrows():
    print(f"  {row['category']:18s} {row['size']:>12s}  {row['filename']}")
print("===========================================")

# horizontal bar — file sizes (log scale)
fig, ax = plt.subplots(figsize=(10, max(3, len(df_files) * 0.4)))
ax.barh(df_files["filename"], df_files["size_bytes"],
        color=[CATEGORY_COLORS[c] for c in df_files["category"]])
ax.set_xscale("log")
ax.set_xlabel("size (bytes, log scale)")
ax.set_title("File sizes on disk")
ax.invert_yaxis()
cats_present = df_files["category"].unique()
ax.legend(handles=[Patch(facecolor=CATEGORY_COLORS[c], label=c) for c in cats_present],
          loc="lower right")
plt.tight_layout(); plt.show()

# # pie — disk usage by category
# cat_sizes = df_files.groupby("category")["size_bytes"].sum().sort_values(ascending=False)
# fig, ax = plt.subplots(figsize=(6, 6))
# ax.pie(cat_sizes.values, labels=cat_sizes.index,
#        colors=[CATEGORY_COLORS[c] for c in cat_sizes.index],
#        autopct=lambda p: f"{p:.1f}%\n({fmt_size(p * cat_sizes.sum() / 100)})",
#        startangle=90)
# ax.set_title("Disk usage by category"); plt.tight_layout(); plt.show()

total = df_files["size_bytes"].sum()
display(Markdown(f"""
**Summary** — files: `{len(df_files)}`, total: `{fmt_size(total)}`, fresh: `{not cached_before}`
"""))

In [ ]:
display(Markdown("## Part 2 — Load into RAM stage"))
display(Markdown("_Disk → RAM. What is read, in what order, how much memory is consumed?_"))

from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@              PART 02: LOAD INTO RAM STAGE                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

ram_baseline = ram_mb()
display(Markdown(f"**Baseline RAM (before model load):** `{ram_baseline:.1f} MB`"))

print("========== STEP 1: BASELINE RAM ==========")
print(f"RAM before any load : {ram_baseline:.1f} MB")
print("==========================================")

log_start = len(STAGE_LOG)

print()
print("========== STEP 2: LOAD TOKENIZER ==========")
print("reads : tokenizer.json, tokenizer_config.json, special_tokens_map.json")
with Stage("load tokenizer", reads="tokenizer.json, tokenizer_config.json, special_tokens_map.json"):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("tokenizer class :", type(tokenizer).__name__)
print("vocab size      :", tokenizer.vocab_size)
print("============================================")

print()
print("========== STEP 3: LOAD MODEL CONFIG ==========")
print("reads : config.json")
with Stage("load model config", reads="config.json"):
    config = AutoConfig.from_pretrained(MODEL_ID)
print("config class :", type(config).__name__)
print("architecture :", config.architectures[0] if config.architectures else "n/a")
print("===============================================")

print()
print("========== STEP 4: LOAD MODEL WEIGHTS ==========")
print("reads : *.safetensors")
with Stage("load model weights", reads="*.safetensors"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    model.eval()
print("model class : ", type(model).__name__)
print("model dtype : ", next(model.parameters()).dtype)
print("model device: ", next(model.parameters()).device)
print("================================================")

df_load = pd.DataFrame(STAGE_LOG[log_start:])
display(Markdown("**Step-by-step load log**"))
display(df_load)

# RAM growth bar
labels  = ["baseline"] + df_load["stage"].tolist()
values  = [ram_baseline] + df_load["ram_after_MB"].tolist()
deltas  = [0] + df_load["ram_delta_MB"].tolist()
colors  = ["#7f7f7f", "#2ca02c", "#1f77b4", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, values, color=colors)
for bar, d in zip(bars, deltas):
    h = bar.get_height()
    txt = f"{h:.0f} MB" + (f"\n(+{d:.0f})" if d > 0 else "")
    ax.text(bar.get_x() + bar.get_width()/2, h, txt, ha="center", va="bottom", fontsize=9)
ax.set_ylabel("RAM (MB)"); ax.set_title("RAM growth across load steps")
ax.set_ylim(0, max(values) * 1.15)
plt.tight_layout(); plt.show()

# time per step
fig, ax = plt.subplots(figsize=(10, 3))
ax.barh(df_load["stage"], df_load["time_s"], color=["#2ca02c", "#1f77b4", "#d62728"])
for i, v in enumerate(df_load["time_s"]):
    ax.text(v, i, f"  {v:.2f} s", va="center")
ax.set_xlabel("seconds"); ax.set_title("Time per load step")
plt.tight_layout(); plt.show()

total_params = sum(p.numel() for p in model.parameters())
param_bytes  = sum(p.numel() * p.element_size() for p in model.parameters())

print()
print("========== STEP 5: MODEL SUMMARY ==========")
print(f"architecture     : {config.architectures[0] if config.architectures else 'n/a'}")
print(f"hidden size      : {config.hidden_size}")
print(f"num layers       : {config.num_hidden_layers}")
print(f"vocab size       : {tokenizer.vocab_size}")
print(f"dtype            : {next(model.parameters()).dtype}")
print(f"device           : {next(model.parameters()).device}")
print(f"total parameters : {total_params:,}  ({total_params / 1e9:.3f} B)")
print(f"theoretical size : {fmt_size(param_bytes)}")
print(f"RAM by model     : {(ram_mb() - ram_baseline):.1f} MB")
print("===========================================")

display(Markdown(f"""
**Model summary**

| property | value |
|---|---|
| architecture     | `{config.architectures[0] if config.architectures else 'n/a'}` |
| hidden size      | `{config.hidden_size}` |
| num layers       | `{config.num_hidden_layers}` |
| vocab size       | `{tokenizer.vocab_size}` |
| dtype            | `{next(model.parameters()).dtype}` |
| device           | `{next(model.parameters()).device}` |
| total parameters | `{total_params:,}` ({total_params / 1e9:.3f} B) |
| theoretical size | `{fmt_size(param_bytes)}` |
| RAM by model     | `{(ram_mb() - ram_baseline):.1f} MB` |
"""))

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 3: expose embed as a separate step)
# STEP 1 — tokenize (text → token IDs, transparent)
# Unchanged from Level 2. Repeated here so each level notebook is self-contained.
# ─────────────────────────────────────────────────────────────

display(Markdown("## Part 3 — Inference (Level 3: expose embed as a separate step)"))
display(Markdown("### Step 1 — tokenize (BPE segment + ID map)"))
display(Markdown("_Unchanged from Level 2. Text → integer token IDs via `{vocab}`._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 1: tokenize — Level 3              @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
text = "The capital of France is"

print("========== IN ==========")
print("text         :", repr(text))
print("  type       :", type(text).__name__)
print("  len (chars):", len(text))
print("========================")
print()

# ─── PROCESS (transparent) ───────────────────────────────────
vocab_size_tok = tokenizer.vocab_size
vocab_size_cfg = config.vocab_size

print("========== PROCESS (transparent) ==========")
print("step                 : tokenizer(text, return_tensors='pt')")
print("op                   : BPE segment + ID map  (string -> sub-word pieces -> integer IDs)")
print("static used          : {vocab}  -- BPE merge rules + token->ID table inside the tokenizer")
print("  tokenizer class    :", type(tokenizer).__name__)
print("  vocab_size (tok)   :", vocab_size_tok,  "  <- base BPE entries")
print("  vocab_size (cfg)   :", vocab_size_cfg,  "  <- model embedding rows (incl. special tokens)")
print("  fast tokenizer     :", tokenizer.is_fast)
print("note                 : NO learned weights. Deterministic. Reversible via decode.")
print("===========================================")
print()

# ─── run tokenize ────────────────────────────────────────────
inputs         = tokenizer(text, return_tensors="pt")
input_ids      = inputs.input_ids           # shape [1, N_tok]
attention_mask = inputs.attention_mask
token_ids_1d   = input_ids[0]               # shape [N_tok]
N_tok          = token_ids_1d.numel()

raw_tokens = tokenizer.convert_ids_to_tokens(token_ids_1d.tolist())
print("========== INTERNAL TOKENIZATION (for inspection) ==========")
print(f"N_tok = {N_tok}")
df_tokens = pd.DataFrame({
    "position": list(range(N_tok)),
    "token_id": token_ids_1d.tolist(),
    "raw_bpe":  [repr(t) for t in raw_tokens],
    "decoded":  [repr(tokenizer.decode([i])) for i in token_ids_1d.tolist()],
})
print(df_tokens.to_string(index=False))
print("============================================================")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("token_ids    :", token_ids_1d.tolist())
print("  shape      :", tuple(token_ids_1d.shape), " <- [N_tok], 1D vector")
print("  dtype      :", token_ids_1d.dtype)
print("  ndim       :", token_ids_1d.ndim)
print("  numel      :", token_ids_1d.numel())
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 1"))

df_step1 = pd.DataFrame([
    {"side": "IN",      "name": "text",                              "value": repr(text),
     "shape": "—", "dtype": "str", "ndim": "—", "dim": "string"},
    {"side": "PROCESS", "name": "tokenizer (BPE segment + ID map)",  "value": f"{type(tokenizer).__name__}, vocab_size={vocab_size_tok}",
     "shape": "—", "dtype": "—",   "ndim": "—", "dim": "lookup (no math)"},
    {"side": "OUT",     "name": "token_ids",                         "value": str(token_ids_1d.tolist()),
     "shape": str(tuple(token_ids_1d.shape)), "dtype": str(token_ids_1d.dtype),
     "ndim": token_ids_1d.ndim, "dim": "1D vector [N_tok]"},
]).set_index("side")
display(df_step1)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 3)
# STEP 2 — embed (NEW: token IDs → vectors)
#
# Internals are NOT yet exposed. We just acknowledge that the
# LLM core cannot consume integer IDs — something turns each ID
# into a vector of size d. The matrix {E} that does this is
# named but not opened (that's Level 4).
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 2 — embed (row lookup in {E}, internals opaque)"))
display(Markdown("_The LLM core does not consume integer IDs. Before it runs, each token ID is converted to a vector of size `d`. The operation is a row lookup in a learned matrix `{E}` of shape `[vocab_size × d]`, but at this level we treat `{E}` as a black box (opened in Level 4)._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 2: embed (opaque) — Level 3        @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("token_ids    :", token_ids_1d.tolist())
print("  shape      :", tuple(token_ids_1d.shape), " <- [N_tok], 1D vector")
print("  dtype      :", token_ids_1d.dtype)
print("  ndim       :", token_ids_1d.ndim)
print("  numel      :", token_ids_1d.numel())
print("========================")
print()

# ─── PROCESS (opaque — only metadata of the embed module) ────
# In a Llama model, the embedding module lives at model.model.embed_tokens.
# We call it directly so embed becomes a visible, standalone step.
embed_module = model.model.embed_tokens
d            = config.hidden_size              # per-token vector dim
vocab_rows   = embed_module.num_embeddings     # rows of {E} = vocab_size (model side)
embed_dtype  = embed_module.weight.dtype
embed_device = embed_module.weight.device

print("========== PROCESS (opaque) ==========")
print("step             : model.model.embed_tokens(token_ids)")
print("op               : row lookup in {E}  (internals opaque at Level 3)")
print("static used      : {embed internals}  -- the learned matrix {E} (not opened yet)")
print("  module class   :", type(embed_module).__name__)
print("  vocab rows     :", vocab_rows,  "  <- rows of {E}")
print("  d (hidden_size):", d,           "  <- columns of {E} = per-token vector dim")
print("  dtype          :", embed_dtype)
print("  device         :", embed_device)
print("note             : we DO NOT inspect {E} values at this level.")
print("                   We only call the module and observe shapes in / out.")
print("                   {E} is opened in Level 4.")
print("======================================")
print()

# ─── run the embed step ──────────────────────────────────────
# input must be on the same device as the embedding module
token_ids_dev = token_ids_1d.to(embed_device)
with torch.no_grad():
    vectors_batched = embed_module(token_ids_dev.unsqueeze(0))   # [1, N_tok, d]
vectors = vectors_batched[0]                                     # [N_tok, d]

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("vectors           : <tensor>")
print("  shape           :", tuple(vectors.shape), " <- [N_tok, d], 2D matrix")
print("  dtype           :", vectors.dtype)
print("  ndim            :", vectors.ndim)
print("  numel           :", vectors.numel())
print(f"  value summary   : min={vectors.min().item():+.4f}  "
      f"max={vectors.max().item():+.4f}  "
      f"mean={vectors.mean().item():+.4f}  "
      f"std={vectors.std().item():.4f}")
print()
print("first 6 dims of each token's vector (for inspection):")
preview = pd.DataFrame(
    vectors[:, :6].float().cpu().numpy(),
    index=[f"tok{i} (id={token_ids_1d[i].item()})" for i in range(N_tok)],
    columns=[f"d{j}" for j in range(6)],
)
print(preview.round(4).to_string())
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 2"))

df_step2 = pd.DataFrame([
    {"side": "IN",      "name": "token_ids",                "value": str(token_ids_1d.tolist()),
     "shape": str(tuple(token_ids_1d.shape)), "dtype": str(token_ids_1d.dtype),
     "ndim": token_ids_1d.ndim, "dim": "1D vector [N_tok]"},
    {"side": "PROCESS", "name": "embed_tokens (opaque)",    "value": f"{type(embed_module).__name__}, rows={vocab_rows}, d={d}",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "opaque"},
    {"side": "OUT",     "name": "vectors",                  "value": f"min={vectors.min().item():+.3f} max={vectors.max().item():+.3f}",
     "shape": str(tuple(vectors.shape)), "dtype": str(vectors.dtype),
     "ndim": vectors.ndim, "dim": "2D matrix [N_tok × d]"},
]).set_index("side")
display(df_step2)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import torch

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 3)
# STEP 3 — LLM core (opaque, but smaller than before)
#
# Difference vs Level 2:
#   - Input is now `vectors [N_tok, d]`, not `token_ids [N_tok]`.
#   - Embedding is NO LONGER inside the box (Step 2 owns it).
#   - {E} is excluded from {core internals}.
#
# Because we feed pre-computed vectors, we can no longer use
# model.generate(input_ids=...). We pass `inputs_embeds=` directly
# to model(...) and pick argmax of the last position's logits.
# (Argmax is the simplest pick policy; it will become a visible
#  step from Level 5 onwards. Here it stays inside the opaque box.)
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 3 — LLM core (opaque, now consumes vectors)"))
display(Markdown("_Same opaque box idea as before, but now embedding is outside. Input is `vectors [N_tok × d]`. True output is still a single integer token ID. Embedding matrix `{E}` is no longer part of `{core internals}`._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 3: LLM core (opaque) — Level 3     @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("vectors           : <tensor>")
print("  shape           :", tuple(vectors.shape), " <- [N_tok, d], 2D matrix")
print("  dtype           :", vectors.dtype)
print("  ndim            :", vectors.ndim)
print("  numel           :", vectors.numel())
print("========================")
print()

# ─── PROCESS (opaque — core only, embedding is now outside) ──
embed_params = embed_module.weight.numel()
total_params = sum(p.numel() for p in model.parameters())
core_params  = total_params - embed_params
core_bytes   = sum(p.numel() * p.element_size() for p in model.parameters()) \
             - embed_module.weight.numel() * embed_module.weight.element_size()
model_dtype  = next(model.parameters()).dtype
model_device = next(model.parameters()).device

print("========== PROCESS (opaque) ==========")
print("step             : model(inputs_embeds=vectors, ...) then argmax over last-position logits")
print("op               : opaque  (transformer blocks + final norm + LM head + pick, all hidden)")
print("static used      : {core internals}  -- learned weights EXCLUDING the embedding matrix {E}")
print("  architecture   :", type(model).__name__)
print("  total params   : {:,}".format(total_params))
print("  embed params   : {:,}  <- excluded from core at this level".format(embed_params))
print("  core params    : {:,}  ({:.3f} B)".format(core_params, core_params/1e9))
print("  core bytes     :", fmt_size(core_bytes))
print("  dtype          :", model_dtype)
print("  device         :", model_device)
print("note             : we still DO NOT look inside the core.")
print("                   Difference vs Level 2: embedding {E} is now OUTSIDE the box.")
print("                   We pass pre-computed vectors via `inputs_embeds`.")
print("======================================")
print()

# ─── run the opaque core ─────────────────────────────────────
# attention_mask is still required so positions are interpreted correctly.
with torch.no_grad():
    outputs = model(
        inputs_embeds=vectors.unsqueeze(0),         # [1, N_tok, d]
        attention_mask=attention_mask.to(model_device),
        use_cache=False,
    )
# outputs.logits has shape [1, N_tok, vocab_size]; pick argmax of the LAST position.
# Logits / softmax / argmax are all still inside the opaque box at Level 3.
last_logits         = outputs.logits[0, -1]                  # [vocab_size]
new_token_id_tensor = torch.argmax(last_logits)              # scalar
new_token_id_int    = int(new_token_id_tensor.item())

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("new_token_id (tensor) :", new_token_id_tensor)
print("  shape               :", tuple(new_token_id_tensor.shape), " <- empty tuple = scalar (0-d)")
print("  dtype               :", new_token_id_tensor.dtype)
print("  ndim                :", new_token_id_tensor.ndim)
print("new_token_id (int)    :", new_token_id_int)
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 3"))

df_step3 = pd.DataFrame([
    {"side": "IN",      "name": "vectors",                       "value": f"min={vectors.min().item():+.3f} max={vectors.max().item():+.3f}",
     "shape": str(tuple(vectors.shape)), "dtype": str(vectors.dtype),
     "ndim": vectors.ndim, "dim": "2D matrix [N_tok × d]"},
    {"side": "PROCESS", "name": "model core (opaque)",           "value": f"{type(model).__name__}, core_params={core_params:,}, dtype={model_dtype}",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "opaque"},
    {"side": "OUT",     "name": "new_token_id",                  "value": str(new_token_id_int),
     "shape": str(tuple(new_token_id_tensor.shape)), "dtype": str(new_token_id_tensor.dtype),
     "ndim": new_token_id_tensor.ndim, "dim": "scalar (0-D)"},
]).set_index("side")
display(df_step3)

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

# ─────────────────────────────────────────────────────────────
# PART 3 — INFERENCE (Level 3)
# STEP 4 — decode (inverse vocab lookup: token ID → text piece)
#
# IDENTICAL to Level 2 / Step 3.
# Same op, same static {vocab}, same inverse lookup.
# Step number shifted from 3 → 4 because embed was promoted to its own step.
# ─────────────────────────────────────────────────────────────

display(Markdown("### Step 4 — decode (inverse vocab lookup)"))
display(Markdown("_Unchanged from Level 2. Pure table lookup in `{vocab}`. Renumbered 3 → 4 because embed is now a separate step._"))

print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")
print("@        PART 03 / STEP 4: decode — Level 3                @")
print("@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@")

# ─── IN ──────────────────────────────────────────────────────
print("========== IN ==========")
print("new_token_id (tensor) :", new_token_id_tensor)
print("  shape               :", tuple(new_token_id_tensor.shape))
print("  dtype               :", new_token_id_tensor.dtype)
print("  ndim                :", new_token_id_tensor.ndim)
print("new_token_id (int)    :", new_token_id_int)
print("========================")
print()

# ─── PROCESS (transparent) ───────────────────────────────────
vocab_size_tok = tokenizer.vocab_size
vocab_size_cfg = config.vocab_size

print("========== PROCESS (transparent) ==========")
print("step                 : tokenizer.decode(new_token_id)")
print("op                   : inverse vocab lookup  (integer index -> token bytes -> UTF-8 string)")
print("static used          : {vocab}  -- SAME BPE table used by tokenize in Step 1")
print("  tokenizer class    :", type(tokenizer).__name__)
print("  vocab_size (tok)   :", vocab_size_tok,  "  <- base BPE entries")
print("  vocab_size (cfg)   :", vocab_size_cfg,  "  <- model embedding rows (incl. special tokens)")
print("  fast tokenizer     :", tokenizer.is_fast)
print("note                 : NO learned weights. Pure deterministic table lookup. Reversible.")
print("===========================================")
print()

token_str_raw = tokenizer.convert_ids_to_tokens(new_token_id_int)
text_piece    = tokenizer.decode([new_token_id_int])

print("========== INTERNAL VOCAB ROW (for inspection) ==========")
print("raw BPE token        :", repr(token_str_raw),
      "  <- 'Ġ' means leading space in GPT-style BPE")
print("decoded text piece   :", repr(text_piece))
print("=========================================================")
print()

# ─── OUT ──────────────────────────────────────────────────────
print("========== OUT ==========")
print("text_piece           :", repr(text_piece))
print("  type               :", type(text_piece).__name__)
print("  len (chars)        :", len(text_piece))
print("=========================")

# ─── Rich view ───────────────────────────────────────────────
display(Markdown("---"))
display(Markdown("### Rich view — Step 4"))

df_step4 = pd.DataFrame([
    {"side": "IN",      "name": "new_token_id",                       "value": str(new_token_id_int),
     "shape": str(tuple(new_token_id_tensor.shape)), "dtype": str(new_token_id_tensor.dtype),
     "ndim": new_token_id_tensor.ndim, "dim": "scalar (0-D)"},
    {"side": "PROCESS", "name": "tokenizer.decode (inverse vocab lookup)",
     "value": f"{type(tokenizer).__name__}, vocab_size={vocab_size_tok}",
     "shape": "—", "dtype": "—", "ndim": "—", "dim": "lookup (no math)"},
    {"side": "OUT",     "name": "text_piece",                         "value": repr(text_piece),
     "shape": "—", "dtype": "str", "ndim": "—", "dim": "string"},
]).set_index("side")
display(df_step4)

# ─── Final end-to-end summary (Level 3 pipeline: 4 steps) ────
display(Markdown("---"))
display(Markdown("### Level 3 pipeline — 4 steps"))

df_pipeline = pd.DataFrame([
    {"step": "tokenize", "op": "BPE segment + ID map",
     "in":  f"text  {repr(text)}",
     "static": "{vocab}  (BPE table)",
     "out": f"token_ids  {tuple(token_ids_1d.shape)}  {token_ids_1d.dtype}",
     "dim_out": "1D vector"},
    {"step": "embed",    "op": "row lookup in {E}  (opaque)",
     "in":  f"token_ids  {tuple(token_ids_1d.shape)}  {token_ids_1d.dtype}",
     "static": "{embed internals}  (the {E} matrix — not opened yet)",
     "out": f"vectors  {tuple(vectors.shape)}  {vectors.dtype}",
     "dim_out": "2D matrix"},
    {"step": "LLM core", "op": "opaque",
     "in":  f"vectors  {tuple(vectors.shape)}  {vectors.dtype}",
     "static": "{core internals}  (learned weights minus {E})",
     "out": f"new_token_id  scalar  {new_token_id_tensor.dtype}",
     "dim_out": "scalar"},
    {"step": "decode",   "op": "inverse vocab lookup",
     "in":  f"new_token_id  scalar  {new_token_id_tensor.dtype}",
     "static": "{vocab}  (BPE table — same as Step 1)",
     "out": f"text_piece  {repr(text_piece)}",
     "dim_out": "string"},
])
df_pipeline.index = range(1, len(df_pipeline) + 1)
df_pipeline.index.name = "#"
display(df_pipeline)